# gf_mul Prediction Matrices (§3.3 Building Prediction Matrices)

학습된 LDA classifier(`.joblib`)를 불러와 **학습에 사용되지 않은 트레이스**에 적용, 6종 (Value/HW × Op0/Op1/Out) 의 **prediction matrix** 를 만들어 HDF5 로 저장한다.

## 파이프라인
```
  weights_gf_mul_{BOARD}_{N_TRACES}.joblib         (from gf_mul_lda.ipynb)
                ↓                ↓
          (clf, mu, sd)    split_seed
                ↓                ↓
       ┌────── 트레이스 HDF5 ──────┐
       │   학습에 안 쓴 부분만 사용 │
       └────────────┬─────────────┘
                    ↓ clf.predict_proba
         prediction matrix (N_POOL, n_classes)
                    ↓
   predict_matrix/pred_matrix_gf_mul_{PLATFORM}_{N_POOL}_seed{POOL_SEED}.h5
```


## 1. Imports

In [ ]:
import h5py
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

## 2. Configuration

In [ ]:
# ===== 사용자 입력 =====
# 파일명에 들어갈 식별자
PLATFORM  = "F303"
N_POOL    = 100000    # prediction matrix 크기 (pool에서 샘플링할 트레이스 수)
POOL_SEED = 12    # pool 샘플링 seed (재현성)

# ===== 입력: 학습된 weights =====
WEIGHTS_PATH = "/Users/ijaeyeon/chipwhisperer/jupyter/user/weights/weights_gf_mul_F303_500000_seed12.joblib"

# ===== 입력: 학습 시 썼던 트레이스 HDF5 =====
# (weight meta 에 기록된 data_h5 를 그대로 사용하는 게 보통)
DATA_H5 = "/Users/ijaeyeon/chipwhisperer/jupyter/user/batches_gf_mul_CW308_STM32F3/500000trace_1400point_seed12.h5"

# ===== 출력: prediction matrix HDF5 =====
OUT_DIR  = "/Users/ijaeyeon/chipwhisperer/jupyter/user/predict_matrix"
OUT_PATH = f"{OUT_DIR}/pred_matrix_gf_mul_{PLATFORM}_{N_POOL}_seed{POOL_SEED}.h5"

# Auto-detect dataset keys inside DATA_H5 (None = auto)
TRACE_KEY = None
A_KEY = None
B_KEY = None
V_KEY = None

print(f"PLATFORM      = {PLATFORM}")
print(f"N_POOL        = {N_POOL}")
print(f"POOL_SEED     = {POOL_SEED}")
print(f"WEIGHTS_PATH  = {WEIGHTS_PATH}")
print(f"DATA_H5       = {DATA_H5}")
print(f"OUT_PATH      = {OUT_PATH}")

## 3. H5 key resolver & helpers

In [ ]:
def resolve_h5_keys(f, trace_key=None, a_key=None, b_key=None, v_key=None):
    keys = list(f.keys())

    def pick(user_key, candidates, name):
        if user_key is not None:
            if user_key not in keys:
                raise KeyError(f"{name} key '{user_key}' not found. Available: {keys}")
            return user_key
        for cand in candidates:
            if cand in keys:
                return cand
        raise KeyError(f"Could not auto-detect {name} key. Available: {keys}")

    trace_key = pick(trace_key, ["traces", "trace", "waves", "wave"], "trace")
    a_key     = pick(a_key,     ["a", "op0", "operand0"], "a")
    b_key     = pick(b_key,     ["b", "op1", "operand1"], "b")
    v_key     = pick(v_key,     ["v", "out", "output"],   "v")
    return trace_key, a_key, b_key, v_key

def hw8(x):
    x = x.astype(np.uint8)
    return np.unpackbits(x[:, None], axis=1).sum(axis=1).astype(np.uint8)

## 4. Load weights

In [ ]:
w = joblib.load(WEIGHTS_PATH)
meta = w["meta"]
print("--- weights meta ---")
for k, v in meta.items():
    print(f"  {k:<12} : {v}")

# 실제 학습에 쓴 분할 정보 (필수)
SPLIT_SEED_TRAIN = int(meta["split_seed"])
TRAIN_RATIO      = float(meta["train_ratio"])
VAL_RATIO        = float(meta["val_ratio"])
N_TRACES_TRAINED = int(meta["n_traces"])

print()
print(f"Template accuracies (stored):")
for k in ("op0_value", "op1_value", "out_value", "op0_hw", "op1_hw", "out_hw"):
    e = w[k]
    print(f"  {k:<10} acc_val={e['acc_val']:.4f}  acc_test={e['acc_test']:.4f}")

## 5. Load traces

In [ ]:
with h5py.File(DATA_H5, "r") as f:
    tk, ak, bk, vk = resolve_h5_keys(f, TRACE_KEY, A_KEY, B_KEY, V_KEY)
    traces_all = f[tk][:].astype(np.float64)
    a_all = f[ak][:].astype(np.uint8)
    b_all = f[bk][:].astype(np.uint8)
    v_all = f[vk][:].astype(np.uint8)

print("traces_all:", traces_all.shape)

# 학습 때 사용한 총 트레이스 수만큼 잘라 쓰기 (meta 기준)
assert N_TRACES_TRAINED <= len(traces_all), \
    f"DATA_H5 에 {len(traces_all)} 개뿐인데 meta.n_traces 는 {N_TRACES_TRAINED} 입니다."
traces_all = traces_all[:N_TRACES_TRAINED]
a_all = a_all[:N_TRACES_TRAINED]
b_all = b_all[:N_TRACES_TRAINED]
v_all = v_all[:N_TRACES_TRAINED]

print(f"Using first {N_TRACES_TRAINED} traces (matching training scope).")

## 6. Reconstruct non-train indices

LDA 학습 때와 **같은 `split_seed` + `train_ratio`** 를 재현하여 `train` 에 들어간 인덱스를 식별하고, **그 외 부분만 pool 후보**로 둔다.

이렇게 해야 prediction matrix 가 학습 데이터와 겹치지 않아 
실제 공격 상황(학습과 무관한 트레이스)을 시뮬레이션할 수 있다.

In [ ]:
# LDA 노트북의 split_train_val_test 와 동일한 로직
np.random.seed(SPLIT_SEED_TRAIN)
idx_permuted = np.random.permutation(N_TRACES_TRAINED)

n_train = int(N_TRACES_TRAINED * TRAIN_RATIO)
n_val   = int(N_TRACES_TRAINED * VAL_RATIO)

idx_train   = idx_permuted[:n_train]
idx_nontrain = idx_permuted[n_train:]   # val + test (학습에 쓰이지 않음)

print(f"train    : {len(idx_train)}")
print(f"non-train: {len(idx_nontrain)}  (pool 후보)")

assert N_POOL <= len(idx_nontrain), \
    f"N_POOL({N_POOL}) > non-train({len(idx_nontrain)}). N_POOL 을 줄이거나 더 많이 수집하세요."

## 7. Sample pool indices

In [ ]:
rng = np.random.default_rng(POOL_SEED)
pool_local = rng.choice(len(idx_nontrain), size=N_POOL, replace=False)
idx_pool = idx_nontrain[pool_local]

print("idx_pool shape:", idx_pool.shape)
print("idx_pool[:10] :", idx_pool[:10])

# pool 트레이스/레이블 추출
X_pool = traces_all[idx_pool]
a_pool = a_all[idx_pool]
b_pool = b_all[idx_pool]
v_pool = v_all[idx_pool]
a_hw_pool = hw8(a_pool)
b_hw_pool = hw8(b_pool)
v_hw_pool = hw8(v_pool)

print("X_pool:", X_pool.shape)

## 8. Build prediction matrices

6개 template 각각에 대해:
1. 저장된 `mu`, `sd` 로 pool 트레이스 z-score 정규화
2. `clf.predict_proba(X_z)` → 확률분포
3. (학습 시 못 본 클래스가 있으면) 전체 `n_classes` 차원으로 0-padding 정렬
4. (pred_matrix, true_labels) 로 결과 저장

In [ ]:
TARGETS = [
    ("op0_value", a_pool,   256),
    ("op1_value", b_pool,   256),
    ("out_value", v_pool,   256),
    ("op0_hw",    a_hw_pool, 9),
    ("op1_hw",    b_hw_pool, 9),
    ("out_hw",    v_hw_pool, 9),
]

def make_prediction_matrix(entry, X, y, n_classes):
    """apply stored clf/mu/sd → (pred_matrix, true_labels, acc_top1)"""
    mu = entry["mu"]
    sd = entry["sd"]
    clf = entry["clf"]

    X_z = (X - mu) / sd
    P = clf.predict_proba(X_z)   # (N, len(clf.classes_))

    # n_classes 폭으로 0-padding
    pred_matrix = np.zeros((len(X), n_classes), dtype=np.float32)
    class_ids = clf.classes_.astype(int)
    pred_matrix[:, class_ids] = P.astype(np.float32)

    top1 = pred_matrix.argmax(axis=1)
    acc_top1 = float((top1 == y).mean())
    return pred_matrix, y.astype(np.uint8), acc_top1

results = {}
for name, y_labels, nc in tqdm(TARGETS, desc="Predicting"):
    pm, tl, acc = make_prediction_matrix(w[name], X_pool, y_labels, nc)
    results[name] = {
        "pred_matrix": pm,
        "true_labels": tl,
        "acc_top1":    acc,
        "n_classes":   nc,
    }
    print(f"  {name:<10}  shape={pm.shape}  acc_top1={acc:.4f}")

## 9. Summary table

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "template":        name,
        "pred_matrix":     r["pred_matrix"].shape,
        "true_labels":     r["true_labels"].shape,
        "acc_top1 (pool)": r["acc_top1"],
    })
pd.DataFrame(rows)

## 10. Save to HDF5

In [ ]:
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

with h5py.File(OUT_PATH, "w") as f:
    # 파일 수준 메타데이터
    f.attrs["platform"]       = PLATFORM
    f.attrs["n_pool"]         = int(N_POOL)
    f.attrs["pool_seed"]      = int(POOL_SEED)
    f.attrs["weights_path"]   = str(WEIGHTS_PATH)
    f.attrs["data_h5"]        = str(DATA_H5)
    f.attrs["split_seed"]     = int(SPLIT_SEED_TRAIN)
    f.attrs["n_traces_train"] = int(N_TRACES_TRAINED)
    f.attrs["train_ratio"]    = float(TRAIN_RATIO)
    f.attrs["val_ratio"]      = float(VAL_RATIO)

    # 각 template 그룹
    for name, r in results.items():
        g = f.create_group(name)
        g.create_dataset("pred_matrix", data=r["pred_matrix"], compression="gzip")
        g.create_dataset("true_labels", data=r["true_labels"], compression="gzip")
        g.attrs["acc_top1"]  = r["acc_top1"]
        g.attrs["n_classes"] = int(r["n_classes"])

    # pool 에 쓰인 원본 인덱스도 참고용으로 기록
    f.create_dataset("idx_pool", data=idx_pool.astype(np.int64), compression="gzip")

print(f"Saved: {OUT_PATH}")
size_mb = Path(OUT_PATH).stat().st_size / (1024 ** 2)
print(f"File size: {size_mb:.2f} MB")

## 11. Verify — reload & inspect

In [ ]:
with h5py.File(OUT_PATH, "r") as f:
    print("=== file attrs ===")
    for k, v in f.attrs.items():
        print(f"  {k:<16}: {v}")
    print()
    print("=== groups ===")
    for k in f.keys():
        if k == "idx_pool":
            ds = f[k]
            print(f"  /{k}  shape={ds.shape} dtype={ds.dtype}")
            continue
        g = f[k]
        pm = g['pred_matrix']
        tl = g['true_labels']
        acc = g.attrs['acc_top1']
        print(
            f"  /{k:<10}  pred_matrix={pm.shape} {pm.dtype}, "
            f"true_labels={tl.shape} {tl.dtype}, acc_top1={acc:.4f}"
        )


### 11-1. Row sum check — 확률분포가 정규화되어 있는지

In [ ]:
with h5py.File(OUT_PATH, "r") as f:
    for k in f.keys():
        if k == "idx_pool":
            continue
        pm = f[k]["pred_matrix"][:100]
        s = pm.sum(axis=1)
        print(f"  /{k:<10}  row_sum  min={s.min():.4f}  mean={s.mean():.4f}  max={s.max():.4f}")

## 12. Usage preview — 다음 단계 공격에서 이렇게 샘플링한다

In [ ]:
# 예: HQC128 codeword 바이트 1개에 대해 n-k=30 개 독립 예측을 꺼내기
with h5py.File(OUT_PATH, "r") as f:
    pm = f["op0_value"]["pred_matrix"][:]
    tl = f["op0_value"]["true_labels"][:]

rng = np.random.default_rng(0)
sampled = rng.choice(len(pm), size=30, replace=False)
print("30 preds shape :", pm[sampled].shape)        # (30, 256)
print("top-1 match   :", (pm[sampled].argmax(1) == tl[sampled]).mean())